# 06 - Surface Construction and Smoothing

Goal: aggregate point estimates into a smooth surface over $(\tau, m)$.

Objective (penalized least squares):
$$
\min_{b(\tau,m)} \sum_g w_g\left(\hat b_g - b(\tau_g,m_g)\right)^2 + \lambda \|\nabla^2 b\|^2
$$
with practical monotonic smoothing and clipping constraints in this demo implementation.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
inp = Path('stage5_rn.csv')
if not inp.exists():
    raise FileNotFoundError('Run notebook 05 first to generate stage5_rn.csv')

df = pd.read_csv(inp)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)
df.head()

,timestamp,p_clipped,x_hat,sigma2_b,sigma2_j,lambda_jump,mu_rn,x_hat_rn,kalman_var_rn,jump_prob,is_jump_dominant
0,2025-11-14 16:31:16+00:00,0.977495,3.771256,0.0,0.0,0.05,-4.774939e-11,3.771256,9.999000e-09,0.05,False


## Build coordinates and raw belief-vol estimates

For this event-stream tech demo, we proxy:
- $\tau$: normalized time-to-end over sample window
- $m$: logit coordinate $x_t$
- $\hat b$: local rolling std of $\Delta x_t$

In [3]:
x = df['x_hat_rn'].to_numpy(float)
dx = np.diff(x, prepend=x[0])

t0, t1 = df['timestamp'].iloc[0], df['timestamp'].iloc[-1]
total_s = max((t1 - t0).total_seconds(), 1.0)
tau = 1.0 - (df['timestamp'] - t0).dt.total_seconds().to_numpy() / total_s
tau = np.clip(tau, 0.0, 1.0)
m = x

b_hat = pd.Series(dx).rolling(60, min_periods=10).std().fillna(method='bfill').to_numpy()
b_hat = np.clip(b_hat, 1e-4, np.nanpercentile(b_hat, 99))

surf_df = pd.DataFrame({'tau': tau, 'm': m, 'b_hat': b_hat})
surf_df.head()

TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

## Grid aggregation and smoothing

In [4]:
tau_bins = np.linspace(0.0, 1.0, 30)
m_bins = np.quantile(surf_df['m'], np.linspace(0.0, 1.0, 35))

surf_df['tau_bin'] = pd.cut(surf_df['tau'], bins=tau_bins, include_lowest=True)
surf_df['m_bin'] = pd.cut(surf_df['m'], bins=np.unique(m_bins), include_lowest=True)

grid = surf_df.pivot_table(index='tau_bin', columns='m_bin', values='b_hat', aggfunc='mean')
grid = grid.interpolate(axis=0).interpolate(axis=1)

# Simple smoothers (row/col rolling means)
grid_s = grid.copy()
grid_s = grid_s.rolling(window=3, min_periods=1, axis=0).mean()
grid_s = grid_s.rolling(window=3, min_periods=1, axis=1).mean()
grid_s = grid_s.clip(lower=0.0)

grid_s.iloc[:5, :5]

NameError: name 'surf_df' is not defined

## Plot and export

In [5]:
plt.figure(figsize=(12, 5))
plt.imshow(grid_s.values, aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='belief-vol estimate')
plt.title('Smoothed Belief-Vol Surface (tau x m bins)')
plt.xlabel('m bins')
plt.ylabel('tau bins')
plt.tight_layout()
plt.show()

NameError: name 'grid_s' is not defined

<Figure size 1200x500 with 0 Axes>

In [6]:
grid_s.to_csv('stage6_surface_grid.csv')
surf_df.to_csv('stage6_surface_points.csv', index=False)
print('saved grid:', Path('stage6_surface_grid.csv').resolve())
print('saved points:', Path('stage6_surface_points.csv').resolve())

NameError: name 'grid_s' is not defined